In [29]:
from openai import OpenAI
import os
import json
import subprocess
import gradio as gr

In [30]:
#llm init in agent
def model_init(api_key,base_url)->object:
    model=OpenAI(api_key=api_key,base_url=base_url)
    return model

In [ ]:
#tools
WORKSPACE="workspace"
def read_file(path) -> str:
    path = os.path.join(WORKSPACE, path)
    with open(path, "r") as file:
        text = file.read()

    return text

def write_file(path, code) -> None:
    path = os.path.join(WORKSPACE, path)
    with open(path, "w") as file:
        file.write(code)


def list_files(path=".") -> list:
    path = os.path.join(WORKSPACE,path)
    return os.listdir(path)


def run_command(command) -> dict:
    result = subprocess.run(
        command,
        shell=True,
        capture_output=True,
        text=True,
        cwd=WORKSPACE
    )

    return {
        "stdout": result.stdout,
        "stderr": result.stderr,
        "returncode": result.returncode
    }

In [32]:
tools = [
    {
        "type": "function",
        "function": {
            "name": "read_file",
            "description": "Read the contents of a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The path of the file to read."
                    }
                },
                "required": ["path"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "write_file",
            "description": "Write code or text to a file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The path of the file to write."
                    },
                    "code": {
                        "type": "string",
                        "description": "The code or text to write into the file."
                    }
                },
                "required": ["path", "code"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "list_files",
            "description": "List all files and folders inside a directory.",
            "parameters": {
                "type": "object",
                "properties": {
                    "path": {
                        "type": "string",
                        "description": "The directory path to list. Defaults to the current directory."
                    }
                },
                "required": []
            }
        }
    },
    {
    "type": "function",
    "function": {
        "name": "run_command",
        "description": "Run a shell command and return its output.",
        "parameters": {
            "type": "object",
            "properties": {
                "command": {
                    "type": "string",
                    "description": "The shell command to execute."
                }
            },
            "required": ["command"]
        }
    }
  }
]

In [33]:
def get_tool(tool_name,args):
    tool_dic={
        "read_file":read_file,
        "write_file":write_file,
        "list_files":list_files,
        "run_command":run_command
    }

    return tool_dic[tool_name](**args)

def tool_handler(msg):
    responses=[]
    for tool_call in msg.tool_calls:
        tool_name=tool_call.function.name
        args=json.loads(tool_call.function.arguments)
        result=get_tool(tool_name=tool_name,args=args)

        responses.append({
            "role":"tool",
            "content":json.dumps(result),
            "tool_call_id":tool_call.id
        })
    return responses

In [34]:
def message():
    sys_msg="""you are a coding agent.
    you modify the code according to the user request.
    before testing check if the required compiler is installed with run_command tool, if its not installed, you do not test it,
    and tell the user clearly that you changed the code but couldn't test it.
    if the compiler is installed you test after you modify the code with run command.
    if an error accures you inspect the error and fix it.
    you send the code that has been modified.
    keep your words simple and precise"""

    msg=[{"role":"system","content":sys_msg}]
    return msg

In [35]:
def coder(messages,model,model_name,reasoning="low"):
    response=model.chat.completions.create(messages=messages,model=model_name,reasoning_effort=reasoning,tools=tools)

    while True:
        msg=response.choices[0].message
        if not msg.tool_calls:
            print(msg.content)
            return msg.content 
        messages.append(msg.model_dump(exclude_none=True))
        tool_responses=tool_handler(msg)
        messages.extend(tool_responses)

        response=model.chat.completions.create(messages=messages,model=model_name,reasoning_effort=reasoning,tools=tools)

In [ ]:
model=model_init("gmm","http://localhost:11434/v1")
def chat(msg,history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages=message()

    messages = messages + history + [{"role": "user", "content": msg}]
    response = coder(messages, model, model_name="deepseek-r1:1.5b ")

    history.append({"role": "user", "content": msg})
    history.append({"role": "assistant", "content": response})
    print (history)
    return history

In [ ]:
def interface(workspace=".")->None:

    with gr.Blocks() as app:

        with gr.Row():

            with gr.Column(scale=1):
                gr.Markdown("Files")

                files = gr.FileExplorer(
                    glob="**/*",
                    root_dir=workspace,
                    file_count="multiple",
                    label="Project"
                )

            with gr.Column(scale=3):

                chatbot = gr.Chatbot(
                    label="agent",
                    sanitize_html=False,
                    render_markdown=False,
                    allow_tags=False,
                )

                message = gr.Textbox(
                    placeholder="type the file and the change you want in it"
                )

                send = gr.Button("send")

        send.click(
            chat,
            inputs=[message, chatbot],
            outputs=chatbot
        )

        message.submit(
            chat,
            inputs=[message, chatbot],
            outputs=chatbot
        )

    app.launch()

In [38]:
interface("workspace")

* Running on local URL:  http://127.0.0.1:7865
* To create a public link, set `share=True` in `launch()`.




Let me help you create a simple "make example.cpp" program that makes the example.cpp say "Hello, agent!". I'll provide the code in a C++ format, assuming you want to make the example executable and run it to see if it works.

However, before testing, make sure you have a C++ compiler installed. I'll show you how to create a simple executable file and run it.

1. First, let's create an example `.h` header file for the "example.cpp" file:

```cpp
// filename: example.h
//

#include <vector>
#include <algorithm>

// Function prototype
void make() {
    // Your source code here
}
//

std::string makeExample = "Your program here";
std::string agentOutput = " agent output here";

// end of .h
```

2. Now, let's create your example `.cpp` source file:

```cpp
// filename: example.cpp
#include <vector>
#include <algorithm>

#include "example.h"

void makeExample() {
    // Declare and initialize the program vars

    std::string program = "Hello, agent!";
    std::string variables = "Hello,